# Unblinded Relationship-Discovery Evaluation

This notebook answers the pre-specified question: **do embedding-selected firms contain more filing-supported, economically meaningful channel relationships than matched controls?** It first freezes the conservative outcome rule while origin remains hidden. Only then does it load the restricted mapping. This ordering prevents knowledge of candidate source from changing adjudication.

In [ ]:
from pathlib import Path
import importlib,sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
ROOT=Path.cwd()
if ROOT.name!='geopolitical_exposure': ROOT=ROOT/'geopolitical_exposure'
sys.path.insert(0,str(ROOT))
PUBLIC=ROOT/'retrieval_benchmark/public'
TRUTH=ROOT/'retrieval_benchmark/restricted_truth'
RESULTS=ROOT/'retrieval_benchmark/evaluation'
RESULTS.mkdir(parents=True,exist_ok=True)
import src.blinded_relationship_evaluation as evaluator
importlib.reload(evaluator)
pd.set_option('display.max_colwidth',160)

## 1. Freeze admissible outcomes before unblinding

A positive requires both a model-supported relationship and passage-level machine validity. Invalid evidence is conservatively counted as unsupported. The rule does not use source group or event returns.

In [ ]:
frozen,manifest=evaluator.freeze_blinded_outcomes(PUBLIC)
display(pd.DataFrame([{'responses':len(frozen),'supported':int(frozen.adjudicated_supported.sum()),'unsupported':int((~frozen.adjudicated_supported).sum()),'machine_valid':int(frozen.machine_valid.sum()),'manifest_sha256':manifest['sha256']}]))
display(frozen.loc[frozen.adjudicated_supported,['blind_id','relationship_type','relationship_direction','channel_relevance','confidence','evidence_quote']])

## 2. Unblind only after the outcome file is frozen

The restricted file reveals whether a candidate was selected by embedding similarity or was its industry-and-size-matched control. It was not loaded during retrieval, API classification, or adjudication.

In [ ]:
panel=evaluator.unblind(PUBLIC,TRUTH)
panel.to_parquet(RESULTS/'unblinded_relationship_panel.parquet',index=False)
summary=evaluator.source_summary(panel)
channels=evaluator.channel_summary(panel)
tests=evaluator.inferential_tests(panel)
summary.to_parquet(RESULTS/'source_summary.parquet',index=False)
channels.to_parquet(RESULTS/'channel_summary.parquet',index=False)
tests.to_parquet(RESULTS/'inferential_tests.parquet',index=False)
display(summary)
display(tests)

## 3. Channel-level decomposition

Each channel contains ten embedding candidates and ten matched controls. With sparse positives, channel estimates are descriptive; the paired overall test is the primary comparison.

In [ ]:
display(channels.sort_values(['channel','hidden_source']))
plot=channels.pivot(index='channel',columns='hidden_source',values='support_rate').fillna(0)
ax=plot[['embedding','matched_control']].plot(kind='bar',figsize=(11,5),color=['#1f4e79','#b8b8b8'])
ax.set_ylabel('Filing-supported relationship rate');ax.set_xlabel('');ax.set_ylim(0,max(.1,float(plot.max().max())*1.25));ax.legend(title='Candidate source')
for container in ax.containers: ax.bar_label(container,labels=[f'{v:.0%}' for v in container.datavalues],padding=3)
plt.xticks(rotation=25,ha='right');plt.tight_layout();plt.savefig(RESULTS/'channel_support_rates.png',dpi=180,bbox_inches='tight');plt.show()

## 4. Inspect the admitted relationships after unblinding

This table shows exactly which relationships drive the comparison. It prevents a statistically convenient aggregate from obscuring weak or economically irrelevant individual cases.

In [ ]:
positive_columns=['blind_id','hidden_source','channel','retrieval_rank','firm_name','ticker','sic2','relationship_type','relationship_direction','channel_relevance','confidence','evidence_quote','reasoning_summary']
positives=panel.loc[panel.adjudicated_supported,positive_columns].sort_values(['hidden_source','channel','retrieval_rank'])
positives.to_csv(RESULTS/'admitted_relationships.csv',index=False)
display(positives)

## 5. Failure analysis: text opportunity and control matching

A zero embedding yield could be mechanical if embedding candidates had less source text. We therefore compare document availability, passage counts, retrieved characters, and retrieval scores across groups. We also report the quality of the industry-and-size matching used to create the counterfactual controls.

In [ ]:
retrieval_detail,retrieval_summary=evaluator.retrieval_diagnostics(PUBLIC,panel)
matching_overall,matching_channels=evaluator.matching_diagnostics(TRUTH)
retrieval_detail.to_parquet(RESULTS/'retrieval_opportunity_detail.parquet',index=False)
retrieval_summary.to_parquet(RESULTS/'retrieval_opportunity_summary.parquet',index=False)
matching_overall.to_parquet(RESULTS/'matching_quality_overall.parquet',index=False)
matching_channels.to_parquet(RESULTS/'matching_quality_by_channel.parquet',index=False)
display(retrieval_summary)
display(matching_overall)
display(matching_channels)

## 6. Evidence-strength sensitivity

The primary rule admits three exact-evidence relationships. Two are weaker economically: National Bank Holdings reports only 0.7% oil-and-gas loans, while Werner describes indirect diesel-cost sensitivity. The strict sensitivity retains only the clearly material, dedicated energy-lending disclosure by Commerce Bancshares. The rejected Granite construction response is also shown as a deliberately permissive upper bound. These rules are disclosed case-by-case rather than optimized statistically.

In [ ]:
sensitivity=[]
rules={
 'primary_exact_evidence':set(panel.loc[panel.adjudicated_supported,'blind_id']),
 'strict_material_direct_disclosure':{'GEO-055'},
 'permissive_include_rejected_granite':set(panel.loc[panel.adjudicated_supported,'blind_id'])|{'GEO-004'},
}
for rule,accepted in rules.items():
    temp=panel.assign(supported_sensitivity=panel.blind_id.isin(accepted))
    row={'rule':rule,'total_supported':int(temp.supported_sensitivity.sum())}
    for source in ['embedding','matched_control']:
        group=temp[temp.hidden_source.eq(source)]
        row[f'{source}_supported']=int(group.supported_sensitivity.sum())
        row[f'{source}_rate']=float(group.supported_sensitivity.mean())
    sensitivity.append(row)
sensitivity=pd.DataFrame(sensitivity)
sensitivity.to_parquet(RESULTS/'evidence_strength_sensitivity.parquet',index=False)
display(sensitivity)

## 7. Interpretation and conclusion

The groups had equal filing availability, equal passage counts, and essentially equal retrieved text volume, so unequal opportunity to observe evidence does not explain the result. Under the primary rule, embedding candidates produced zero supported relationships and controls produced three; under the strict rule the comparison is zero versus one. Thus the conclusion is stable to evidence strength.

This test does **not** support incremental relationship discovery by holdings embeddings for the five Russia–Ukraine channels in this sample. It does not establish that embeddings are universally uninformative, causal exposure is absent, or a trading strategy cannot work. Holdings similarity may encode investor clientele, latent risk, or relationships not disclosed in the retrieved 10-K passages. Statistical power is low because only three primary positives were found. The appropriate report is therefore a transparent negative result, not a claim of equivalence.

A higher embedding support rate would have shown incremental discovery only within this frozen design. Confirmation in another event, a richer corpus such as earnings calls, or verified supply-chain data would be required before making a broader conclusion.